<a href="https://colab.research.google.com/github/borgesjose/Grover_Algorithm/blob/main/Algoritmo__de__Grover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Algoritmo de Grover**

Nesta atividade, são implementados os códigos do algoritmo de grover em Python, utilizando o ambiente Jupyter (collab).

Aluno: José Borges do Carmo Neto; https://github.com/borgesjose

Link do github: [Notebook Salvo]()

Adotaremos a notação da base computacional $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, para descrever a evolução dos circuitos passo a passo.*texto em itálico*

In [6]:
!pip install qiskit qiskit-aer -q

In [13]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

import numpy as np

## Criar  PORTA Z multicontrolada:

In [10]:
def apply_multi_controlled_z(qc, qubits):
  if len(qubits) == 1:
        qc.z(qubits[0])
        return
  controls = qubits[:-1]
  target = qubits[-1]
  qc.h(target)
  qc.mcx(controls, target)
  qc.h(target)

In [11]:
def build_oracle(n, targets):
    qc = QuantumCircuit(n, name="Oraculo")
    for t in targets:
        bits = format(t, f"0{n}b")   # ex: n=3, t=5 -> '101'
        # descobrir quais posicoes de qubit (0 a n-1) tem bit '0' no alvo,
        # lembrando da inversao little-endian (bits[n-1-i] corresponde a qubit[i])
        qubits_to_flip = []
        for i in range(n):
            if bits[n - 1 - i] == '0':
                qc.x(i)
                qubits_to_flip.append(i)

        # aplicar apply_multi_controlled_z(qc, list(range(n)))
        apply_multi_controlled_z(qc, list(range(n)))

        # desfazer os X (mesma lista de antes)
        for i in qubits_to_flip:
            qc.x(i)
    return qc

In [14]:
qc_test = QuantumCircuit(3)
qc_test.x(0)   # prepara |001> manualmente (qubit 0 = 1, resto = 0)
qc_test.compose(build_oracle(3, [1]), inplace=True)  # alvo = 1 = |001>
print(Statevector(qc_test))

Statevector([ 0.00000000e+00+0.j, -1.00000000e+00+0.j,  0.00000000e+00+0.j,
              0.00000000e+00+0.j,  0.00000000e+00+0.j,  2.23711432e-17+0.j,
              0.00000000e+00+0.j,  0.00000000e+00+0.j],
            dims=(2, 2, 2))


In [15]:
def build_diffuser(n):
    qc = QuantumCircuit(n, name="Difusor")
    # H em todos os qubits
    qc.h(range(n))
    # X em todos os qubits
    qc.x(range(n))
    # apply_multi_controlled_z nos n qubits
    apply_multi_controlled_z(qc, list(range(n)))
    # X em todos os qubits
    qc.x(range(n))
    # H em todos os qubits
    qc.h(range(n))

    return qc

In [17]:
import numpy as np

def num_iterations(n, k):
    N = 2 ** n
    r = np.floor(np.pi / 4 * np.sqrt(N / k))
    return int(r)

def grover_circuit(n, targets, r=None):
    k = len(targets)
    if r is None:
        r = num_iterations(n, k)
    qc = QuantumCircuit(n, n)
    # H em todos os qubits (superposicao inicial)
    qc.h(range(n))
    oracle = build_oracle(n, targets)
    diffuser = build_diffuser(n)
    for _ in range(r):
        # aplicar oracle no circuito (use qc.compose(oracle, range(n), inplace=True))
        qc.compose(oracle, range(n), inplace=True)
        # aplicar diffuser da mesma forma
        qc.compose(diffuser, range(n), inplace=True)
    # medir todos os qubits nos bits classicos correspondentes
    qc.measure(range(n), range(n))
    return qc, r

### Cenario 1


In [18]:
qc1, r1 = grover_circuit(2, [3])
print("r =", r1)

sim = AerSimulator()
result = sim.run(qc1, shots=1024).result()
counts = result.get_counts()
print(counts)

r = 1
{'11': 1024}


### Cenario 2

In [19]:
import time

def classical_linear_search(N, targets):
    targets_set = set(targets)
    t0 = time.perf_counter()
    consultas = 0
    encontrado = None
    for x in range(N):
        consultas += 1
        if x in targets_set:
            encontrado = x
            break
    t1 = time.perf_counter()
    return consultas, t1 - t0, encontrado

In [20]:
n2 = 16
alvo2 = 2**n2 - 1   # |111...1>

t0 = time.perf_counter()
qc2, r2 = grover_circuit(n2, [alvo2])
sim = AerSimulator()
result2 = sim.run(qc2, shots=1024).result()
t1 = time.perf_counter()

counts2 = result2.get_counts()
print(f"r = {r2} iteracoes")
print(f"tempo quantico (simulacao): {t1 - t0:.4f} s")
print("estado mais frequente:", max(counts2, key=counts2.get))

consultas_c, tempo_c, encontrado_c = classical_linear_search(2**n2, [alvo2])
print(f"\nbusca classica: {consultas_c} consultas, {tempo_c:.6f} s")

r = 201 iteracoes
tempo quantico (simulacao): 2.6323 s
estado mais frequente: 1111111111111111

busca classica: 65536 consultas, 0.004521 s


### Cenario 3

In [ ]:
for n_teste in [16, 18, 20, 22]:
    alvo = 2**n_teste - 1
    print(f"\nTestando n={n_teste} ({2**n_teste} estados)...")
    try:
        t0 = time.perf_counter()
        qc_teste, r_teste = grover_circuit(n_teste, [alvo])
        sim = AerSimulator()
        result_teste = sim.run(qc_teste, shots=256).result()
        t1 = time.perf_counter()
        counts_teste = result_teste.get_counts()
        acerto = max(counts_teste, key=counts_teste.get) == format(alvo, f"0{n_teste}b")
        print(f"  r={r_teste} | tempo={t1-t0:.2f}s | acertou={acerto}")
    except Exception as e:
        print(f"  FALHOU: {type(e).__name__}: {e}")
        break


Testando n=16 (65536 estados)...
  r=201 | tempo=2.61s | acertou=True

Testando n=18 (262144 estados)...
  r=402 | tempo=24.20s | acertou=True

Testando n=20 (1048576 estados)...
  r=804 | tempo=197.93s | acertou=True

Testando n=22 (4194304 estados)...
